# Metinsiz sayfalari OCR ile okunabilir yapma

Bazi kaynak kitaplarin bir kismi taranmis goruntu ve metin katmani yok;
o sayfalardan Drive hicbir icerik cikaramiyor. Bu defter once hangi
sayfalarin metinsiz oldugunu olcer, sonra sadece onlari Turkce OCR ile
okunabilir hale getirir.

Hucreleri sirayla calistir. 3. hucrenin ciktisina gore 4. hucredeki
`OCRLANACAK` listesini duzenle.


In [ ]:
# 1) Drive'i bagla
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2) Araclari kur (~1-2 dk)
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-tur ocrmypdf > /dev/null
!pip install -q pymupdf
print("kurulum tamam")


In [ ]:
# 3) TANI: hangi sayfalarda metin katmani yok?
#
# Once olcelim. Kitabin tamamini tarar, her sayfanin cikarilabilir metin
# uzunlugunu bakar ve metinsiz araliklari toplu halde yazdirir.

import fitz

KITAP = "/content/drive/Othercomputers/iMac'im/Desktop/KBB/1KBB Otoloji.pdf"
ESIK  = 50          # bu karakterden az metin = "metin katmani yok" sayilir

kitap = fitz.open(KITAP)
n = len(kitap)
bos = [s for s in range(1, n + 1) if len(kitap[s - 1].get_text().strip()) < ESIK]

print(f"{KITAP.split('/')[-1]}: {n} sayfa, {len(bos)} sayfada metin yok "
      f"(%{100 * len(bos) / n:.0f})\n")

# Ardisik sayfalari aralik olarak topla
araliklar = []
for s in bos:
    if araliklar and s == araliklar[-1][1] + 1:
        araliklar[-1][1] = s
    else:
        araliklar.append([s, s])

print("Metinsiz araliklar (PDF sayfa numaralari):")
for a, b in araliklar:
    print(f"  {a:4d} - {b:4d}   ({b - a + 1} sayfa)")

print("\nBu araliklari asagidaki hucrede OCRLANACAK listesine yaz.")


In [ ]:
# 4) OCR: metinsiz sayfalari okunabilir hale getir
#
# Secilen araligi ayri bir PDF'e cikarir, Turkce OCR ile metin katmani
# ekler ve KBB_kaynak klasorune yazar. Ciktinin adi normal parcalarla ayni
# desende, sonuna -OCR eklenmis olur.

import fitz, os, subprocess

KITAP  = "/content/drive/Othercomputers/iMac'im/Desktop/KBB/1KBB Otoloji.pdf"
HEDEF  = "/content/drive/MyDrive/KBB_kaynak"
SINIR  = 45 * 1024 * 1024

OCRLANACAK = [(281, 400)]      # 3. hucrenin ciktisina gore duzenle

os.makedirs(HEDEF, exist_ok=True)
ad = os.path.splitext(os.path.basename(KITAP))[0]
kitap = fitz.open(KITAP)

for bas, son in OCRLANACAK:
    print(f"\n{bas}-{son} arasi isleniyor...")

    ham = f"/content/ham-{bas:04d}-{son:04d}.pdf"
    yeni = fitz.open()
    yeni.insert_pdf(kitap, from_page=bas - 1, to_page=son - 1)
    yeni.save(ham)
    yeni.close()
    print(f"  cikarildi: {os.path.getsize(ham)/1e6:.0f} MB")

    # --force-ocr: mevcut (bozuk/eksik) metin katmanini yok sayip yeniden uretir
    # --optimize 3 ve --jpeg-quality: dosyayi kucultur, OCR metnini etkilemez
    okunur = f"/content/ocr-{bas:04d}-{son:04d}.pdf"
    print("  OCR calisiyor, sayfa basina ~2-4 sn...")
    subprocess.run([
        "ocrmypdf", "-l", "tur", "--force-ocr",
        "--optimize", "3", "--jpeg-quality", "70",
        "--output-type", "pdf", ham, okunur
    ], check=True)
    print(f"  OCR bitti: {os.path.getsize(okunur)/1e6:.0f} MB")

    # 45 MB ustundeyse parcala - Drive'in metin cikarma limiti icin
    d = fitz.open(okunur)
    toplam = len(d)
    if os.path.getsize(okunur) <= SINIR:
        parcalar = [(0, toplam - 1)]
    else:
        adim = max(1, int(SINIR * toplam / os.path.getsize(okunur)))
        parcalar = [(i, min(i + adim - 1, toplam - 1)) for i in range(0, toplam, adim)]

    for i, (p0, p1) in enumerate(parcalar):
        cikti = os.path.join(HEDEF, f"{ad}-OCR-{bas+p0:04d}-{bas+p1:04d}.pdf")
        y = fitz.open()
        y.insert_pdf(d, from_page=p0, to_page=p1)
        y.save(cikti)
        y.close()
        mb = os.path.getsize(cikti) / 1e6
        uyari = "   <-- UYARI: 50 MB ustu" if mb > 50 else ""
        print(f"    {os.path.basename(cikti)}  {mb:.1f} MB{uyari}")

    d.close()

print("\nBitti. Parcalar KBB_kaynak klasorunde.")


In [ ]:
# 5) DOGRULAMA: OCR gercekten metin uretti mi?
import fitz, glob, os

for yol in sorted(glob.glob("/content/drive/MyDrive/KBB_kaynak/*-OCR-*.pdf")):
    d = fitz.open(yol)
    ornek = d[len(d) // 2].get_text().strip()
    print(f"\n{os.path.basename(yol)}  ({len(d)} sayfa)")
    print(f"  ortadaki sayfadan {len(ornek)} karakter")
    print("  ilk 200 karakter:", ornek[:200].replace("\n", " ") or "(BOS - OCR basarisiz)")
    d.close()
